# Module 3: Attention & Transformers

**Learning Objectives:**
- Implement scaled dot-product attention from first principles
- Build multi-head attention with clear reshape operations
- Understand and implement positional encodings (sinusoidal, learned, 2D, timestep)
- Implement cross-attention for conditioning on external information
- Build spatial self-attention for images (the exact module used in diffusion U-Nets)
- Understand efficient attention and memory trade-offs
- Assemble full transformer blocks and a minimal Vision Transformer

**Estimated Time:** 2--3 hours

**Key Papers:**
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) -- Vaswani et al. 2017
- [An Image is Worth 16x16 Words (ViT)](https://arxiv.org/abs/2010.11929) -- Dosovitskiy et al. 2020

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
import numpy as np
import math
import time
from typing import Optional

torch.manual_seed(42)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---
## 3.1 -- Dot-Product Attention from Scratch

**Attention as soft lookup.** Think of attention like a database query:

| Component | Database analogy | Shape |
|-----------|-----------------|-------|
| **Query (Q)** | The question you are asking | `(B, seq_q, d_k)` |
| **Key (K)** | The address / index of each entry | `(B, seq_k, d_k)` |
| **Value (V)** | The content stored at each address | `(B, seq_k, d_v)` |

The mechanism:
1. **Dot-product similarity:** `scores = Q @ K^T` -- how well does each query match each key?
2. **Scale:** Divide by `sqrt(d_k)` to prevent softmax saturation when `d_k` is large.
3. **Softmax:** Convert scores to attention weights (each row sums to 1).
4. **Weighted sum:** `output = weights @ V` -- retrieve values weighted by relevance.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

**Why scaling matters:** When `d_k` is large, the dot products grow large in magnitude. This pushes softmax into regions where it has extremely small gradients (near 0 or 1), causing vanishing gradients during training.

### Worked Example: Single-Head Attention from Scratch

In [ ]:
# --- Single-head attention from scratch ---
torch.manual_seed(42)

B, seq_len, d_k = 1, 5, 8  # batch=1, 5 tokens, key dim=8

Q = torch.randn(B, seq_len, d_k)  # (B, seq_len, d_k)
K = torch.randn(B, seq_len, d_k)  # (B, seq_len, d_k)
V = torch.randn(B, seq_len, d_k)  # (B, seq_len, d_k)

# Step 1: Raw dot-product scores
scores = Q @ K.transpose(-2, -1)  # (B, seq_len, seq_len)
print(f"Raw scores shape: {scores.shape}")
print(f"Raw scores range: [{scores.min():.2f}, {scores.max():.2f}]")

# Step 2: Scale
scale = math.sqrt(d_k)
scaled_scores = scores / scale  # (B, seq_len, seq_len)
print(f"\nScaled scores range: [{scaled_scores.min():.2f}, {scaled_scores.max():.2f}]")

# Step 3: Softmax -> attention weights
attn_weights = torch.softmax(scaled_scores, dim=-1)  # (B, seq_len, seq_len)
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Row sums (should be 1.0): {attn_weights[0].sum(dim=-1)}")

# Step 4: Weighted sum of values
output = attn_weights @ V  # (B, seq_len, d_k)
print(f"Output shape: {output.shape}")

In [ ]:
# --- Visualize attention weights ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# With scaling
axes[0].imshow(attn_weights[0].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Attention Weights (with scaling)")
axes[0].set_xlabel("Key position")
axes[0].set_ylabel("Query position")
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f"{attn_weights[0, i, j]:.2f}", ha="center", va="center",
                     color="white" if attn_weights[0, i, j] < 0.5 else "black", fontsize=8)

# Without scaling -- softmax saturates
unscaled_weights = torch.softmax(scores, dim=-1)  # (B, seq_len, seq_len)
axes[1].imshow(unscaled_weights[0].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Attention Weights (WITHOUT scaling)")
axes[1].set_xlabel("Key position")
axes[1].set_ylabel("Query position")
for i in range(seq_len):
    for j in range(seq_len):
        axes[1].text(j, i, f"{unscaled_weights[0, i, j]:.2f}", ha="center", va="center",
                     color="white" if unscaled_weights[0, i, j] < 0.5 else "black", fontsize=8)

plt.tight_layout()
plt.show()

# Demonstrate the saturation problem
print("Entropy of attention weights (higher = more spread out):")
scaled_entropy = -(attn_weights * attn_weights.log()).sum(-1).mean()
unscaled_entropy = -(unscaled_weights * (unscaled_weights + 1e-10).log()).sum(-1).mean()
print(f"  With scaling:    {scaled_entropy:.3f}")
print(f"  Without scaling: {unscaled_entropy:.3f}")
print("Without scaling, attention concentrates on fewer keys (lower entropy).")

### Exercise 3.1: Implement `scaled_dot_product_attention`

Implement the full attention formula with:
1. Proper `1/sqrt(d_k)` scaling
2. An optional attention mask (additive mask: set masked positions to `-inf` before softmax)
3. Verify your output matches `F.scaled_dot_product_attention`

In [ ]:
# Exercise 3.1 -- Your implementation here

def scaled_dot_product_attention(
    Q: torch.Tensor,        # (B, seq_q, d_k)
    K: torch.Tensor,        # (B, seq_k, d_k)
    V: torch.Tensor,        # (B, seq_k, d_v)
    attn_mask: Optional[torch.Tensor] = None,  # (B, seq_q, seq_k) or (seq_q, seq_k)
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Args:
        Q: Query tensor of shape (B, seq_q, d_k)
        K: Key tensor of shape (B, seq_k, d_k)
        V: Value tensor of shape (B, seq_k, d_v)
        attn_mask: Optional additive attention mask. Positions with -inf are ignored.

    Returns:
        output: (B, seq_q, d_v)
        attn_weights: (B, seq_q, seq_k)
    """
    # TODO: implement
    pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def scaled_dot_product_attention(
    Q: torch.Tensor,        # (B, seq_q, d_k)
    K: torch.Tensor,        # (B, seq_k, d_k)
    V: torch.Tensor,        # (B, seq_k, d_v)
    attn_mask: Optional[torch.Tensor] = None,  # (B, seq_q, seq_k) or (seq_q, seq_k)
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Args:
        Q: Query tensor of shape (B, seq_q, d_k)
        K: Key tensor of shape (B, seq_k, d_k)
        V: Value tensor of shape (B, seq_k, d_v)
        attn_mask: Optional additive attention mask. Positions with -inf are ignored.

    Returns:
        output: (B, seq_q, d_v)
        attn_weights: (B, seq_q, seq_k)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (B, seq_q, seq_k)

    if attn_mask is not None:
        scores = scores + attn_mask  # (B, seq_q, seq_k)

    attn_weights = torch.softmax(scores, dim=-1)  # (B, seq_q, seq_k)
    output = attn_weights @ V  # (B, seq_q, d_v)

    return output, attn_weights


# --- Verify against F.scaled_dot_product_attention ---
torch.manual_seed(42)
B, seq_len, d_k = 2, 10, 16
Q = torch.randn(B, seq_len, d_k)
K = torch.randn(B, seq_len, d_k)
V = torch.randn(B, seq_len, d_k)

our_output, our_weights = scaled_dot_product_attention(Q, K, V)
# F.scaled_dot_product_attention expects (B, num_heads, seq, d_k) for 4D or (B, seq, d_k) for 3D
# We use the 3D variant
ref_output = F.scaled_dot_product_attention(Q, K, V)  # (B, seq_len, d_k)

print(f"Our output shape: {our_output.shape}")
print(f"Ref output shape: {ref_output.shape}")
print(f"Max difference: {(our_output - ref_output).abs().max():.2e}")
assert torch.allclose(our_output, ref_output, atol=1e-5), "Outputs do not match!"
print("Outputs match F.scaled_dot_product_attention.")

# --- Test with causal mask ---
causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf")), diagonal=1)  # (seq_len, seq_len)
our_output_masked, our_weights_masked = scaled_dot_product_attention(Q, K, V, attn_mask=causal_mask)
ref_output_masked = F.scaled_dot_product_attention(Q, K, V, attn_mask=causal_mask)

print(f"\nWith causal mask - Max difference: {(our_output_masked - ref_output_masked).abs().max():.2e}")
assert torch.allclose(our_output_masked, ref_output_masked, atol=1e-5), "Masked outputs do not match!"
print("Masked outputs match too.")
print(f"Causal attention weights (row 0 should only attend to position 0):")
print(our_weights_masked[0, :3, :5].detach().numpy().round(3))

---
## 3.2 -- Multi-Head Attention

A single attention head computes one set of attention weights. **Multi-head attention** runs `h` heads in parallel, each with its own learned projections, so different heads can attend to different aspects of the input (e.g., one head for positional relationships, another for semantic similarity).

**Mechanics:**

| Step | Operation | Shape transformation |
|------|-----------|---------------------|
| 1. Project | `Q_proj = X @ W_Q` | `(B, seq, d_model)` --> `(B, seq, d_model)` |
| 2. Split heads | reshape + transpose | `(B, seq, d_model)` --> `(B, h, seq, d_k)` where `d_k = d_model / h` |
| 3. Attention | scaled dot-product per head | `(B, h, seq, d_k)` --> `(B, h, seq, d_k)` |
| 4. Merge heads | transpose + reshape | `(B, h, seq, d_k)` --> `(B, seq, d_model)` |
| 5. Output proj | `output = merged @ W_O` | `(B, seq, d_model)` --> `(B, seq, d_model)` |

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) \, W^O$$

where each $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$.

### Worked Example: MultiHeadAttention from Scratch

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention implemented from scratch.

    Args:
        d_model: Total model dimension.
        num_heads: Number of parallel attention heads.
    """

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Projection matrices (all combined into single linear layers for efficiency)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self,
        query: torch.Tensor,   # (B, seq_q, d_model)
        key: torch.Tensor,     # (B, seq_k, d_model)
        value: torch.Tensor,   # (B, seq_k, d_model)
        attn_mask: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, seq_q, _ = query.shape
        seq_k = key.size(1)

        # 1. Project Q, K, V
        Q = self.W_Q(query)  # (B, seq_q, d_model)
        K = self.W_K(key)    # (B, seq_k, d_model)
        V = self.W_V(value)  # (B, seq_k, d_model)

        # 2. Split into heads: (B, seq, d_model) -> (B, num_heads, seq, d_k)
        Q = Q.view(B, seq_q, self.num_heads, self.d_k).transpose(1, 2)  # (B, h, seq_q, d_k)
        K = K.view(B, seq_k, self.num_heads, self.d_k).transpose(1, 2)  # (B, h, seq_k, d_k)
        V = V.view(B, seq_k, self.num_heads, self.d_k).transpose(1, 2)  # (B, h, seq_k, d_k)

        # 3. Scaled dot-product attention (per head, batched)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, h, seq_q, seq_k)

        if attn_mask is not None:
            scores = scores + attn_mask

        attn_weights = torch.softmax(scores, dim=-1)  # (B, h, seq_q, seq_k)
        attn_output = attn_weights @ V  # (B, h, seq_q, d_k)

        # 4. Merge heads: (B, h, seq_q, d_k) -> (B, seq_q, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, seq_q, self.d_model)

        # 5. Output projection
        output = self.W_O(attn_output)  # (B, seq_q, d_model)

        return output, attn_weights


# --- Test it ---
torch.manual_seed(42)
B, seq_len, d_model, num_heads = 2, 10, 64, 8
x = torch.randn(B, seq_len, d_model)

mha = MultiHeadAttention(d_model, num_heads)
output, weights = mha(x, x, x)  # self-attention: Q=K=V=x

print(f"Input shape:   {x.shape}")           # (2, 10, 64)
print(f"Output shape:  {output.shape}")       # (2, 10, 64)
print(f"Weights shape: {weights.shape}")      # (2, 8, 10, 10)
print(f"Weight row sum (should be 1.0): {weights[0, 0, 0].sum():.4f}")

In [ ]:
# --- Compare to nn.MultiheadAttention ---
torch.manual_seed(42)

ref_mha = nn.MultiheadAttention(d_model, num_heads, bias=False, batch_first=True)

# Copy our weights into the reference module for a fair comparison.
# nn.MultiheadAttention stores W_Q, W_K, W_V as a single in_proj_weight.
with torch.no_grad():
    ref_mha.in_proj_weight.copy_(
        torch.cat([mha.W_Q.weight, mha.W_K.weight, mha.W_V.weight], dim=0)
    )
    ref_mha.out_proj.weight.copy_(mha.W_O.weight)

ref_output, ref_weights = ref_mha(x, x, x)
print(f"Max diff vs nn.MultiheadAttention: {(output - ref_output).abs().max():.2e}")
assert torch.allclose(output, ref_output, atol=1e-5), "Mismatch!"
print("Our MultiHeadAttention matches nn.MultiheadAttention.")

### Exercise 3.2: Multi-Head Attention with Einsum + Memory Profiling

1. Re-implement multi-head attention using `torch.einsum` for the score computation.
2. Profile peak memory usage as sequence length increases -- observe the quadratic scaling.

In [ ]:
# Exercise 3.2 -- Your implementation here

class MultiHeadAttentionEinsum(nn.Module):
    """Multi-head attention using einsum for score computation."""

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(
        self,
        query: torch.Tensor,   # (B, seq_q, d_model)
        key: torch.Tensor,     # (B, seq_k, d_model)
        value: torch.Tensor,   # (B, seq_k, d_model)
    ) -> torch.Tensor:
        # TODO: implement using einsum
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class MultiHeadAttentionEinsum(nn.Module):
    """Multi-head attention using einsum for score computation."""

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self,
        query: torch.Tensor,   # (B, seq_q, d_model)
        key: torch.Tensor,     # (B, seq_k, d_model)
        value: torch.Tensor,   # (B, seq_k, d_model)
    ) -> torch.Tensor:
        B, seq_q, _ = query.shape
        seq_k = key.size(1)

        # Project and reshape to (B, h, seq, d_k)
        Q = self.W_Q(query).view(B, seq_q, self.num_heads, self.d_k).transpose(1, 2)  # (B, h, seq_q, d_k)
        K = self.W_K(key).view(B, seq_k, self.num_heads, self.d_k).transpose(1, 2)    # (B, h, seq_k, d_k)
        V = self.W_V(value).view(B, seq_k, self.num_heads, self.d_k).transpose(1, 2)  # (B, h, seq_k, d_k)

        # Einsum for attention scores: batch, head, query_pos, key_pos
        scores = torch.einsum("bhqd,bhkd->bhqk", Q, K) / math.sqrt(self.d_k)  # (B, h, seq_q, seq_k)

        attn_weights = torch.softmax(scores, dim=-1)  # (B, h, seq_q, seq_k)

        # Einsum for weighted sum
        attn_output = torch.einsum("bhqk,bhkd->bhqd", attn_weights, V)  # (B, h, seq_q, d_k)

        # Merge heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, seq_q, self.d_model)  # (B, seq_q, d_model)
        return self.W_O(attn_output)  # (B, seq_q, d_model)


# --- Verify correctness ---
torch.manual_seed(42)
mha_einsum = MultiHeadAttentionEinsum(d_model=64, num_heads=8)
x_test = torch.randn(2, 10, 64)
out_einsum = mha_einsum(x_test, x_test, x_test)
print(f"Einsum MHA output shape: {out_einsum.shape}")

# --- Memory profiling vs sequence length ---
print("\nMemory profiling: attention matrix size vs sequence length")
print(f"{'Seq len':>10} {'Attn matrix (MB)':>18} {'Ratio vs prev':>15}")

prev_size = None
for seq_len in [64, 128, 256, 512, 1024, 2048]:
    # Attention matrix is (B, h, seq, seq) float32
    B_test, h_test = 1, 8
    attn_matrix_bytes = B_test * h_test * seq_len * seq_len * 4  # float32 = 4 bytes
    attn_matrix_mb = attn_matrix_bytes / (1024 ** 2)
    ratio = attn_matrix_mb / prev_size if prev_size else 1.0
    print(f"{seq_len:>10} {attn_matrix_mb:>15.2f} MB {ratio:>14.1f}x")
    prev_size = attn_matrix_mb

print("\nDoubling sequence length -> 4x memory (quadratic scaling).")

---
## 3.3 -- Positional Encodings

**The problem:** Attention is permutation-invariant. If you shuffle the input sequence, the attention weights change but the mechanism has no inherent notion of order. We must explicitly inject position information.

**Sinusoidal positional encoding** (Vaswani et al. 2017):

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Intuition:** Each dimension oscillates at a different frequency. Low-index dimensions change slowly (long wavelengths for coarse position), high-index dimensions change rapidly (short wavelengths for fine position). Together they create a unique "fingerprint" for each position.

| Encoding type | Pros | Cons |
|--------------|------|------|
| Sinusoidal | Generalizes to unseen lengths, no parameters | Fixed, cannot be task-adapted |
| Learned (`nn.Embedding`) | Task-adaptive | Cannot generalize beyond training length |
| 2D (for images) | Preserves spatial structure | Must handle variable resolutions |

**Connection to diffusion models:** Timestep embeddings in diffusion models use the *exact same sinusoidal formula*, replacing `pos` with the diffusion timestep `t`. This encodes "how noisy is the input" rather than "where in the sequence."

### Worked Example: Sinusoidal Positional Encoding

In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Generate sinusoidal positional encoding.

    Args:
        max_len: Maximum sequence length.
        d_model: Model dimension (must be even).

    Returns:
        pe: Positional encoding tensor of shape (max_len, d_model).
    """
    pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)

    # Compute the division term: 10000^(2i/d_model)
    # Equivalent to exp(-2i * log(10000) / d_model) for numerical stability
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
    )  # (d_model/2,)

    pe[:, 0::2] = torch.sin(position * div_term)  # even indices
    pe[:, 1::2] = torch.cos(position * div_term)  # odd indices

    return pe  # (max_len, d_model)


# --- Visualize ---
pe = sinusoidal_positional_encoding(max_len=128, d_model=64)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of all dimensions
im = axes[0].imshow(pe.numpy().T, aspect="auto", cmap="RdBu", interpolation="nearest")
axes[0].set_xlabel("Position")
axes[0].set_ylabel("Dimension")
axes[0].set_title("Sinusoidal Positional Encoding")
plt.colorbar(im, ax=axes[0])

# Individual dimensions showing different frequencies
for dim_idx in [0, 1, 4, 8, 16, 32]:
    axes[1].plot(pe[:, dim_idx].numpy(), label=f"dim {dim_idx}", alpha=0.8)
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Value")
axes[1].set_title("Individual Dimensions (different frequencies)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Show generalization: encoding for position 200 (beyond max_len=128)
pe_extended = sinusoidal_positional_encoding(max_len=256, d_model=64)
print("Positions 126-130 (around the boundary of the original max_len=128):")
print(pe_extended[126:131, :4].numpy().round(3))
print("The encoding smoothly continues beyond the training length.")

### Exercise 3.3: Positional Encodings -- Sinusoidal, Learned, 2D, and Timestep

1. Implement both sinusoidal and learned positional encodings as `nn.Module`.
2. Implement 2D positional encoding for image patches (separate row and column encodings).
3. Implement sinusoidal **timestep embedding** -- this is used directly in diffusion models.

In [ ]:
# Exercise 3.3 -- Your implementation here

class SinusoidalPE(nn.Module):
    """Sinusoidal positional encoding (no learnable parameters)."""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        # TODO: register pe as a buffer (not a parameter)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, seq_len, d_model) -> add PE -> (B, seq_len, d_model)
        pass


class LearnedPE(nn.Module):
    """Learned positional encoding using nn.Embedding."""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass


class PositionalEncoding2D(nn.Module):
    """2D positional encoding for image patches (separate row + column)."""

    def __init__(self, grid_size: int, d_model: int) -> None:
        super().__init__()
        # TODO: implement -- encode row and column positions separately
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, grid_size*grid_size, d_model)
        pass


class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding for diffusion models.

    Same formula as positional encoding, but applied to scalar timesteps.
    Followed by a small MLP to project to the desired dimension.
    """

    def __init__(self, d_model: int) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: (B,) integer or float timesteps -> (B, d_model)
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class SinusoidalPE(nn.Module):
    """Sinusoidal positional encoding (no learnable parameters)."""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        pe = sinusoidal_positional_encoding(max_len, d_model)  # (max_len, d_model)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        Args:
            x: (B, seq_len, d_model)
        Returns:
            (B, seq_len, d_model)
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]  # (B, seq_len, d_model)


class LearnedPE(nn.Module):
    """Learned positional encoding using nn.Embedding."""

    def __init__(self, max_len: int, d_model: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add learned positional encoding to input.

        Args:
            x: (B, seq_len, d_model)
        Returns:
            (B, seq_len, d_model)
        """
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)  # (seq_len,)
        return x + self.embedding(positions).unsqueeze(0)  # (B, seq_len, d_model)


class PositionalEncoding2D(nn.Module):
    """2D positional encoding for image patches (separate row + column sinusoidal)."""

    def __init__(self, grid_size: int, d_model: int) -> None:
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for row/col split"
        self.grid_size = grid_size
        half_d = d_model // 2

        # Separate sinusoidal encodings for rows and columns
        row_pe = sinusoidal_positional_encoding(grid_size, half_d)  # (grid_size, half_d)
        col_pe = sinusoidal_positional_encoding(grid_size, half_d)  # (grid_size, half_d)

        # Build 2D grid: for each (row, col), concatenate row_pe[row] and col_pe[col]
        pe_2d = torch.zeros(grid_size * grid_size, d_model)  # (grid_size^2, d_model)
        for r in range(grid_size):
            for c in range(grid_size):
                idx = r * grid_size + c
                pe_2d[idx, :half_d] = row_pe[r]
                pe_2d[idx, half_d:] = col_pe[c]

        self.register_buffer("pe_2d", pe_2d.unsqueeze(0))  # (1, grid_size^2, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add 2D positional encoding to patch sequence.

        Args:
            x: (B, grid_size*grid_size, d_model)
        Returns:
            (B, grid_size*grid_size, d_model)
        """
        return x + self.pe_2d  # (B, grid_size^2, d_model)


class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding for diffusion models.

    Same sinusoidal formula as positional encoding, but applied to scalar timesteps.
    Followed by a 2-layer MLP to project to the desired output dimension.
    This is the standard timestep embedding used in DDPM, Stable Diffusion, etc.
    """

    def __init__(self, d_model: int) -> None:
        super().__init__()
        assert d_model % 2 == 0
        self.d_model = d_model
        half_d = d_model // 2

        # Precompute the frequency terms (not learnable)
        div_term = torch.exp(
            torch.arange(0, half_d, dtype=torch.float32) * (-math.log(10000.0) / half_d)
        )  # (half_d,)
        self.register_buffer("div_term", div_term)

        # MLP to project sinusoidal features to d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Linear(d_model * 4, d_model),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """Embed timesteps.

        Args:
            t: (B,) integer or float timesteps
        Returns:
            (B, d_model)
        """
        t = t.float().unsqueeze(-1)  # (B, 1)
        sinusoidal = torch.cat([
            torch.sin(t * self.div_term),  # (B, half_d)
            torch.cos(t * self.div_term),  # (B, half_d)
        ], dim=-1)  # (B, d_model)

        return self.mlp(sinusoidal)  # (B, d_model)


# --- Test all implementations ---
torch.manual_seed(42)

# 1. Sinusoidal PE
sin_pe = SinusoidalPE(max_len=100, d_model=64)
x = torch.randn(2, 20, 64)
out = sin_pe(x)
print(f"SinusoidalPE: input {x.shape} -> output {out.shape}")

# 2. Learned PE
learned_pe = LearnedPE(max_len=100, d_model=64)
out_learned = learned_pe(x)
print(f"LearnedPE: input {x.shape} -> output {out_learned.shape}")
print(f"  Learned PE has {sum(p.numel() for p in learned_pe.parameters())} parameters")

# 3. 2D PE for image patches
pe_2d = PositionalEncoding2D(grid_size=8, d_model=64)
patches = torch.randn(2, 64, 64)  # 8x8 grid = 64 patches
out_2d = pe_2d(patches)
print(f"2D PE: input {patches.shape} -> output {out_2d.shape}")

# Visualize 2D PE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pe_grid = pe_2d.pe_2d[0].view(8, 8, 64)  # (8, 8, 64)
axes[0].imshow(pe_grid[:, :, 0].numpy(), cmap="RdBu")
axes[0].set_title("2D PE -- dim 0 (row encoding)")
axes[1].imshow(pe_grid[:, :, 32].numpy(), cmap="RdBu")
axes[1].set_title("2D PE -- dim 32 (column encoding)")
plt.tight_layout()
plt.show()

# 4. Timestep embedding (diffusion!)
ts_emb = SinusoidalTimestepEmbedding(d_model=128)
timesteps = torch.tensor([0, 100, 500, 999])  # (4,)
emb = ts_emb(timesteps)
print(f"\nTimestep embedding: input {timesteps.shape} -> output {emb.shape}")

# Visualize: different timesteps produce different embeddings
fig, ax = plt.subplots(figsize=(10, 3))
for i, t_val in enumerate(timesteps):
    ax.plot(emb[i].detach().numpy(), label=f"t={t_val.item()}", alpha=0.8)
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Value")
ax.set_title("Sinusoidal Timestep Embeddings (used in diffusion models)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3.4 -- Layer Norm vs Batch Norm in Attention Contexts

| Property | BatchNorm | LayerNorm |
|----------|-----------|-----------|
| Normalizes across | Batch dimension (per channel) | Feature dimension (per sample) |
| Batch dependence | Yes -- needs batch statistics | No -- each sample is independent |
| Variable-length sequences | Problematic | Works naturally |
| Inference behavior | Uses running stats (different from training) | Same at train and inference |
| Used in | CNNs (convolution blocks) | Transformers, attention blocks |

**LayerNorm formula:**
$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where $\mu$ and $\sigma^2$ are computed across the feature dimension (last dim) for each individual sample.

**Pre-norm vs post-norm:** The original Transformer used post-norm (`x + LayerNorm(Attention(x))`). Modern transformers use **pre-norm** (`x + Attention(LayerNorm(x))`) because it leads to more stable training and better gradient flow.

### Worked Example: LayerNorm from Scratch + Pre-Norm Block

In [ ]:
class LayerNormScratch(nn.Module):
    """LayerNorm implemented from scratch.

    Args:
        d_model: Feature dimension to normalize over.
        eps: Small constant for numerical stability.
    """

    def __init__(self, d_model: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable scale
        self.beta = nn.Parameter(torch.zeros(d_model))    # learnable shift

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize across the last dimension.

        Args:
            x: (..., d_model)
        Returns:
            (..., d_model)
        """
        mean = x.mean(dim=-1, keepdim=True)       # (..., 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # (..., 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)   # (..., d_model)
        return self.gamma * x_norm + self.beta              # (..., d_model)


# --- Verify against nn.LayerNorm ---
torch.manual_seed(42)
d_model = 64
x = torch.randn(2, 10, d_model)

our_ln = LayerNormScratch(d_model)
ref_ln = nn.LayerNorm(d_model)

# Copy weights for fair comparison
with torch.no_grad():
    our_ln.gamma.copy_(ref_ln.weight)
    our_ln.beta.copy_(ref_ln.bias)

our_out = our_ln(x)
ref_out = ref_ln(x)

print(f"Max diff vs nn.LayerNorm: {(our_out - ref_out).abs().max():.2e}")
assert torch.allclose(our_out, ref_out, atol=1e-5), "Mismatch!"
print("Our LayerNorm matches nn.LayerNorm.")

# --- Pre-norm attention block ---
print("\n--- Pre-Norm Attention Block ---")
print("Structure: x + Attention(LayerNorm(x))")
print("This is the standard in modern transformers (GPT-2+, ViT, diffusion U-Nets).")

mha_block = MultiHeadAttention(d_model=64, num_heads=8)
ln = nn.LayerNorm(64)

x = torch.randn(2, 10, 64)
x_normed = ln(x)                          # LayerNorm first (pre-norm)
attn_out, _ = mha_block(x_normed, x_normed, x_normed)  # self-attention
out = x + attn_out                         # residual connection

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Residual preserved: input mean={x.mean():.4f}, output mean={out.mean():.4f}")

### Exercise 3.4: Implement LayerNorm, Compare to `nn.LayerNorm`

In [ ]:
# Exercise 3.4 -- Implement LayerNorm and verify

class MyLayerNorm(nn.Module):
    """Your LayerNorm implementation."""

    def __init__(self, normalized_shape: int, eps: float = 1e-5) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class MyLayerNorm(nn.Module):
    """LayerNorm implementation matching nn.LayerNorm behavior."""

    def __init__(self, normalized_shape: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize across the last dimension.

        Args:
            x: (..., normalized_shape)
        Returns:
            (..., normalized_shape)
        """
        mean = x.mean(dim=-1, keepdim=True)                    # (..., 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)      # (..., 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)       # (..., normalized_shape)
        return self.weight * x_norm + self.bias                 # (..., normalized_shape)


# --- Verify ---
torch.manual_seed(42)
d = 128
x = torch.randn(4, 20, d)

my_ln = MyLayerNorm(d)
ref_ln = nn.LayerNorm(d)

with torch.no_grad():
    my_ln.weight.copy_(ref_ln.weight)
    my_ln.bias.copy_(ref_ln.bias)

my_out = my_ln(x)
ref_out = ref_ln(x)

print(f"Max absolute difference: {(my_out - ref_out).abs().max():.2e}")
assert torch.allclose(my_out, ref_out, atol=1e-5)
print("MyLayerNorm matches nn.LayerNorm.")

---
## 3.5 -- Cross-Attention: Conditioning on External Information

| Attention type | Query source | Key/Value source | Use case |
|---------------|-------------|------------------|----------|
| **Self-attention** | Input X | Input X | Tokens attend to each other |
| **Cross-attention** | Input X | Context C | X attends to external info C |

In diffusion models, cross-attention is the primary mechanism for **text-to-image conditioning**:
- **Q** comes from image features (the denoising U-Net's intermediate representations)
- **K, V** come from text encoder output (e.g., CLIP or T5 embeddings)

The formula is identical to self-attention -- the only code difference is that Q comes from one source while K and V come from another.

### Worked Example: Cross-Attention -- Image Features Attend to Text Embeddings

In [ ]:
# --- Cross-Attention: image features attend to text embeddings ---
torch.manual_seed(42)

B = 2
d_model = 64
num_heads = 8

# Simulate image features (e.g., from a U-Net layer at 8x8 resolution)
n_spatial = 8 * 8  # 64 spatial positions
image_features = torch.randn(B, n_spatial, d_model)  # (B, 64, 64)

# Simulate text embeddings (e.g., from CLIP: 77 tokens)
n_text_tokens = 10  # shorter for demo
text_embeddings = torch.randn(B, n_text_tokens, d_model)  # (B, 10, 64)

# Use our MultiHeadAttention -- the ONLY difference from self-attention is the source of K, V
mha = MultiHeadAttention(d_model, num_heads)

# Self-attention: Q=K=V=image_features
self_attn_out, self_attn_weights = mha(image_features, image_features, image_features)

# Cross-attention: Q=image_features, K=V=text_embeddings
cross_attn_out, cross_attn_weights = mha(image_features, text_embeddings, text_embeddings)

print("Self-attention:")
print(f"  Q, K, V all from image: {image_features.shape}")
print(f"  Output: {self_attn_out.shape}")          # (B, 64, 64)
print(f"  Weights: {self_attn_weights.shape}")      # (B, 8, 64, 64) -- image attends to image

print("\nCross-attention:")
print(f"  Q from image: {image_features.shape}")
print(f"  K, V from text: {text_embeddings.shape}")
print(f"  Output: {cross_attn_out.shape}")          # (B, 64, 64) -- same as input
print(f"  Weights: {cross_attn_weights.shape}")      # (B, 8, 64, 10) -- image attends to text

# Visualize cross-attention: which text tokens does each spatial position attend to?
fig, ax = plt.subplots(figsize=(8, 6))
# Average across heads, show first batch element
avg_cross_weights = cross_attn_weights[0].mean(dim=0).detach().numpy()  # (64, 10)
im = ax.imshow(avg_cross_weights, aspect="auto", cmap="viridis")
ax.set_xlabel("Text token index")
ax.set_ylabel("Spatial position (image patch)")
ax.set_title("Cross-Attention: Image patches attending to text tokens")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\nKey insight: the code for cross-attention is identical to self-attention.")
print("The only change is passing different tensors for Q vs K,V.")

### Exercise 3.5: Cross-Attention Module Conditioning Image Features on Class Embeddings

Build a `CrossAttentionBlock` that conditions image features on class embeddings. This previews how text-to-image diffusion works -- the same mechanism is used with text embeddings from CLIP.

In [ ]:
# Exercise 3.5 -- Your implementation here

class CrossAttentionBlock(nn.Module):
    """Cross-attention block: image features attend to conditioning context.

    Args:
        d_model: Feature dimension of image features.
        d_context: Feature dimension of context (class/text embeddings).
        num_heads: Number of attention heads.
    """

    def __init__(self, d_model: int, d_context: int, num_heads: int) -> None:
        super().__init__()
        # TODO: implement
        # Hint: you need separate projections for Q (from image) and K,V (from context)
        # Include LayerNorm and a residual connection
        pass

    def forward(
        self,
        x: torch.Tensor,        # (B, n_spatial, d_model)
        context: torch.Tensor,  # (B, n_context, d_context)
    ) -> torch.Tensor:
        # TODO: implement
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class CrossAttentionBlock(nn.Module):
    """Cross-attention block: image features attend to conditioning context.

    Uses pre-norm with a residual connection.
    Q is projected from image features, K and V from context.

    Args:
        d_model: Feature dimension of image features.
        d_context: Feature dimension of context (class/text embeddings).
        num_heads: Number of attention heads.
    """

    def __init__(self, d_model: int, d_context: int, num_heads: int) -> None:
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_model = d_model

        self.norm = nn.LayerNorm(d_model)

        # Q from image features, K/V from context (may have different d_context)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_context, d_model, bias=False)  # project context to d_model
        self.W_V = nn.Linear(d_context, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,        # (B, n_spatial, d_model)
        context: torch.Tensor,  # (B, n_context, d_context)
    ) -> torch.Tensor:
        """Apply cross-attention with residual connection.

        Args:
            x: Image features (B, n_spatial, d_model)
            context: Conditioning context (B, n_context, d_context)

        Returns:
            (B, n_spatial, d_model)
        """
        B, n_spatial, _ = x.shape
        n_context = context.size(1)

        # Pre-norm
        x_norm = self.norm(x)  # (B, n_spatial, d_model)

        # Project: Q from image, K/V from context
        Q = self.W_Q(x_norm).view(B, n_spatial, self.num_heads, self.d_k).transpose(1, 2)    # (B, h, n_spatial, d_k)
        K = self.W_K(context).view(B, n_context, self.num_heads, self.d_k).transpose(1, 2)   # (B, h, n_context, d_k)
        V = self.W_V(context).view(B, n_context, self.num_heads, self.d_k).transpose(1, 2)   # (B, h, n_context, d_k)

        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, h, n_spatial, n_context)
        attn_weights = torch.softmax(scores, dim=-1)             # (B, h, n_spatial, n_context)
        attn_output = attn_weights @ V                           # (B, h, n_spatial, d_k)

        # Merge heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, n_spatial, self.d_model)  # (B, n_spatial, d_model)
        output = self.W_O(attn_output)  # (B, n_spatial, d_model)

        return x + output  # (B, n_spatial, d_model) -- residual connection


# --- Test: condition image features on class embeddings ---
torch.manual_seed(42)

d_model = 64
d_context = 32    # class embedding dim (different from d_model)
num_heads = 8
num_classes = 10

# Image features from a U-Net layer
image_feats = torch.randn(4, 16 * 16, d_model)  # (B=4, 256 spatial, 64)

# Class embeddings: each class has an embedding, expanded to a single "token"
class_labels = torch.randint(0, num_classes, (4,))  # (4,) random class labels
class_embed_table = nn.Embedding(num_classes, d_context)
class_embs = class_embed_table(class_labels).unsqueeze(1)  # (4, 1, 32) -- single token per class

cross_attn = CrossAttentionBlock(d_model, d_context, num_heads)
conditioned = cross_attn(image_feats, class_embs)

print(f"Image features:  {image_feats.shape}")  # (4, 256, 64)
print(f"Class embeddings: {class_embs.shape}")   # (4, 1, 32)
print(f"Conditioned output: {conditioned.shape}") # (4, 256, 64) -- same shape as input
print(f"\nThis is exactly how diffusion models condition on text/class labels.")

---
## 3.6 -- Spatial Self-Attention for Images

Images are not sequences, but we can reshape them to apply attention:

```
(B, C, H, W)  -->  reshape  -->  (B, H*W, C)  -->  self-attention  -->  (B, H*W, C)  -->  reshape  -->  (B, C, H, W)
```

Each spatial position (pixel or patch) becomes a "token." Attention captures **long-range dependencies** that convolutions (with limited receptive fields) cannot.

**The cost:** The attention matrix is `(H*W) x (H*W)`. For a 256x256 image, that is 65536 x 65536 = over 4 billion entries. This is why diffusion U-Nets only apply attention at lower resolutions:

| Resolution | Sequence length | Attention matrix size | Feasible? |
|-----------|----------------|----------------------|-----------|
| 8 x 8 | 64 | 64 x 64 = 4K | Yes |
| 16 x 16 | 256 | 256 x 256 = 65K | Yes |
| 32 x 32 | 1024 | 1M | Tight |
| 64 x 64 | 4096 | 16M | Expensive |
| 256 x 256 | 65536 | 4B | OOM |

### Worked Example: Feature Map Self-Attention + Quadratic Scaling Demo

In [ ]:
# --- Spatial self-attention: feature map -> sequence -> attention -> reshape back ---
torch.manual_seed(42)

B, C, H, W = 1, 32, 8, 8
feature_map = torch.randn(B, C, H, W)  # (B, C, H, W)

# Step 1: Reshape to sequence
# (B, C, H, W) -> (B, C, H*W) -> (B, H*W, C)
seq = feature_map.flatten(2).transpose(1, 2)  # (B, H*W, C) = (1, 64, 32)
print(f"Feature map: {feature_map.shape} -> Sequence: {seq.shape}")

# Step 2: Self-attention
mha = MultiHeadAttention(d_model=C, num_heads=4)
attn_out, attn_weights = mha(seq, seq, seq)  # (B, H*W, C)
print(f"Attention output: {attn_out.shape}")

# Step 3: Reshape back to spatial
# (B, H*W, C) -> (B, C, H*W) -> (B, C, H, W)
out_map = attn_out.transpose(1, 2).view(B, C, H, W)  # (B, C, H, W)
print(f"Back to feature map: {out_map.shape}")

# --- Measure compute time at different resolutions ---
print("\n--- Quadratic scaling with spatial resolution ---")
print(f"{'Resolution':>12} {'Seq length':>12} {'Time (ms)':>12}")

times = []
resolutions = [4, 8, 16, 32, 64]
seq_lengths = []

for res in resolutions:
    C_test = 32
    x_test = torch.randn(1, res * res, C_test)
    mha_test = MultiHeadAttention(d_model=C_test, num_heads=4)

    # Warmup
    _ = mha_test(x_test, x_test, x_test)

    # Time it
    start = time.perf_counter()
    for _ in range(10):
        _ = mha_test(x_test, x_test, x_test)
    elapsed = (time.perf_counter() - start) / 10 * 1000  # ms

    times.append(elapsed)
    seq_lengths.append(res * res)
    print(f"{res}x{res:>5} {res*res:>12} {elapsed:>11.2f}")

# Plot quadratic scaling
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seq_lengths, times, "o-", label="Measured time")
# Fit quadratic for reference
coeffs = np.polyfit(seq_lengths, times, 2)
x_fit = np.linspace(seq_lengths[0], seq_lengths[-1], 100)
ax.plot(x_fit, np.polyval(coeffs, x_fit), "--", alpha=0.5, label="Quadratic fit")
ax.set_xlabel("Sequence length (H * W)")
ax.set_ylabel("Time (ms)")
ax.set_title("Attention compute time scales quadratically with spatial resolution")
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 3.6: `SpatialSelfAttention(nn.Module)`

Implement a self-contained module that takes `(B, C, H, W)` feature maps, handles the reshaping internally, applies multi-head self-attention, and returns `(B, C, H, W)`. This is the exact module used inside diffusion U-Nets.

In [ ]:
# Exercise 3.6 -- Your implementation here

class SpatialSelfAttention(nn.Module):
    """Self-attention for spatial feature maps used in diffusion U-Nets.

    Takes (B, C, H, W) -> applies self-attention over spatial positions -> returns (B, C, H, W).

    Args:
        channels: Number of channels in the feature map.
        num_heads: Number of attention heads.
    """

    def __init__(self, channels: int, num_heads: int = 4) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        # TODO: reshape -> attention -> reshape back
        # Include group norm, residual connection
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class SpatialSelfAttention(nn.Module):
    """Self-attention for spatial feature maps used in diffusion U-Nets.

    Takes (B, C, H, W) -> applies self-attention over spatial positions -> returns (B, C, H, W).
    Uses GroupNorm (standard in diffusion U-Nets), QKV projections via 1x1 convolutions,
    and a residual connection.

    Args:
        channels: Number of channels in the feature map.
        num_heads: Number of attention heads.
    """

    def __init__(self, channels: int, num_heads: int = 4) -> None:
        super().__init__()
        assert channels % num_heads == 0

        self.channels = channels
        self.num_heads = num_heads
        self.d_k = channels // num_heads

        # GroupNorm is standard in diffusion U-Nets (groups=32 typically, or channels if < 32)
        self.norm = nn.GroupNorm(num_groups=min(32, channels), num_channels=channels)

        # QKV projections as 1x1 convolutions (equivalent to linear on channel dim)
        self.W_Q = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.W_K = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.W_V = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.W_O = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

        # Initialize output projection to zero for stable residual at init
        nn.init.zeros_(self.W_O.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply spatial self-attention with residual connection.

        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        B, C, H, W = x.shape
        residual = x

        # Normalize
        x_norm = self.norm(x)  # (B, C, H, W)

        # Project to Q, K, V using 1x1 convs
        Q = self.W_Q(x_norm)  # (B, C, H, W)
        K = self.W_K(x_norm)  # (B, C, H, W)
        V = self.W_V(x_norm)  # (B, C, H, W)

        # Reshape to sequence: (B, C, H, W) -> (B, C, H*W) -> (B, num_heads, d_k, H*W)
        Q = Q.view(B, self.num_heads, self.d_k, H * W)  # (B, h, d_k, H*W)
        K = K.view(B, self.num_heads, self.d_k, H * W)  # (B, h, d_k, H*W)
        V = V.view(B, self.num_heads, self.d_k, H * W)  # (B, h, d_k, H*W)

        # Attention: transpose to (B, h, H*W, d_k) for standard attention
        Q = Q.permute(0, 1, 3, 2)  # (B, h, H*W, d_k)
        K = K.permute(0, 1, 3, 2)  # (B, h, H*W, d_k)
        V = V.permute(0, 1, 3, 2)  # (B, h, H*W, d_k)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, h, H*W, H*W)
        attn_weights = torch.softmax(scores, dim=-1)             # (B, h, H*W, H*W)
        attn_out = attn_weights @ V                              # (B, h, H*W, d_k)

        # Reshape back: (B, h, H*W, d_k) -> (B, C, H, W)
        attn_out = attn_out.permute(0, 1, 3, 2)            # (B, h, d_k, H*W)
        attn_out = attn_out.contiguous().view(B, C, H, W)  # (B, C, H, W)

        # Output projection + residual
        return residual + self.W_O(attn_out)  # (B, C, H, W)


# --- Test ---
torch.manual_seed(42)
spatial_attn = SpatialSelfAttention(channels=64, num_heads=8)

feature_map = torch.randn(2, 64, 16, 16)  # typical diffusion U-Net resolution
output = spatial_attn(feature_map)

print(f"Input:  {feature_map.shape}")   # (2, 64, 16, 16)
print(f"Output: {output.shape}")         # (2, 64, 16, 16)
print(f"Same shape: {feature_map.shape == output.shape}")
print(f"Parameters: {sum(p.numel() for p in spatial_attn.parameters()):,}")
print(f"\nThis module slots directly into a diffusion U-Net.")

---
## 3.7 -- Efficient Attention and Memory Considerations

Standard attention materializes the full `(seq_len x seq_len)` attention matrix, requiring O(n^2) memory.

**Flash Attention** (Dao et al. 2022) is a fused CUDA kernel that:
1. Never materializes the full attention matrix in HBM (GPU global memory)
2. Computes attention in blocks using SRAM (fast on-chip memory)
3. Reduces memory from O(n^2) to O(n)
4. Is often *faster* despite doing the same FLOPs (IO-aware algorithm)

**In PyTorch:** `F.scaled_dot_product_attention` automatically dispatches to Flash Attention (or memory-efficient attention) when available. You get the speedup for free.

**Practical strategy in diffusion U-Nets:**
- Use attention only at bottleneck / lower resolutions (8x8, 16x16, sometimes 32x32)
- At full resolution (256x256), use only convolutions
- When Flash Attention is unavailable, chunked attention can reduce peak memory

### Worked Example: Naive vs Efficient Attention Memory Comparison

In [ ]:
# --- Compare naive attention vs F.scaled_dot_product_attention ---

def naive_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """Naive attention that materializes the full attention matrix.

    Args:
        Q, K, V: (B, h, seq_len, d_k)
    Returns:
        (B, h, seq_len, d_k)
    """
    d_k = Q.size(-1)
    # This creates the full (seq_len x seq_len) matrix in memory
    attn_matrix = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (B, h, seq, seq) -- O(n^2) memory
    attn_weights = torch.softmax(attn_matrix, dim=-1)         # (B, h, seq, seq)
    return attn_weights @ V                                    # (B, h, seq, d_k)


# Compare memory footprint for different sequence lengths
print("Memory comparison: naive attention matrix vs F.scaled_dot_product_attention")
print(f"{'Seq len':>10} {'Resolution':>12} {'Naive attn matrix':>20} {'Note':>20}")

for seq_len, res_label in [(64, "8x8"), (256, "16x16"), (1024, "32x32"),
                            (4096, "64x64"), (65536, "256x256")]:
    # Attention matrix memory: B * h * seq * seq * 4 bytes (float32)
    B, h = 1, 8
    naive_bytes = B * h * seq_len * seq_len * 4
    naive_mb = naive_bytes / (1024 ** 2)

    note = "feasible" if naive_mb < 500 else "OOM on most GPUs" if naive_mb > 8000 else "expensive"
    print(f"{seq_len:>10} {res_label:>12} {naive_mb:>17.1f} MB {note:>20}")

print("\nF.scaled_dot_product_attention avoids materializing this matrix entirely.")
print("Flash Attention uses O(n) memory regardless of sequence length.")

# Actual timing comparison (at a feasible resolution)
torch.manual_seed(42)
seq_len = 1024  # 32x32 resolution
B, h, d_k = 1, 8, 32
Q = torch.randn(B, h, seq_len, d_k)
K = torch.randn(B, h, seq_len, d_k)
V = torch.randn(B, h, seq_len, d_k)

# Naive
start = time.perf_counter()
for _ in range(20):
    out_naive = naive_attention(Q, K, V)
time_naive = (time.perf_counter() - start) / 20 * 1000

# F.scaled_dot_product_attention
start = time.perf_counter()
for _ in range(20):
    out_sdpa = F.scaled_dot_product_attention(Q, K, V)
time_sdpa = (time.perf_counter() - start) / 20 * 1000

print(f"\nTiming at seq_len={seq_len} (B={B}, h={h}, d_k={d_k}):")
print(f"  Naive:  {time_naive:.2f} ms")
print(f"  SDPA:   {time_sdpa:.2f} ms")
print(f"  Speedup: {time_naive / time_sdpa:.1f}x")
print(f"  Results match: {torch.allclose(out_naive, out_sdpa, atol=1e-4)}")

### Exercise 3.7: Implement Chunked Attention

When Flash Attention is unavailable (e.g., on CPU or older GPUs), chunked attention processes the query in blocks to reduce peak memory. Instead of computing the full `(seq_q x seq_k)` attention matrix at once, we compute it in chunks of `chunk_size` queries at a time.

In [ ]:
# Exercise 3.7 -- Your implementation here

def chunked_attention(
    Q: torch.Tensor,  # (B, h, seq_q, d_k)
    K: torch.Tensor,  # (B, h, seq_k, d_k)
    V: torch.Tensor,  # (B, h, seq_k, d_v)
    chunk_size: int = 128,
) -> torch.Tensor:
    """Compute attention in chunks over the query dimension to reduce peak memory.

    Instead of materializing a (seq_q x seq_k) matrix, materializes (chunk_size x seq_k).

    Args:
        Q: (B, h, seq_q, d_k)
        K: (B, h, seq_k, d_k)
        V: (B, h, seq_k, d_v)
        chunk_size: Number of queries to process at once.

    Returns:
        (B, h, seq_q, d_v)
    """
    # TODO: implement
    pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def chunked_attention(
    Q: torch.Tensor,  # (B, h, seq_q, d_k)
    K: torch.Tensor,  # (B, h, seq_k, d_k)
    V: torch.Tensor,  # (B, h, seq_k, d_v)
    chunk_size: int = 128,
) -> torch.Tensor:
    """Compute attention in chunks over the query dimension to reduce peak memory.

    Peak attention matrix size: (chunk_size x seq_k) instead of (seq_q x seq_k).

    Args:
        Q: (B, h, seq_q, d_k)
        K: (B, h, seq_k, d_k)
        V: (B, h, seq_k, d_v)
        chunk_size: Number of queries to process at once.

    Returns:
        (B, h, seq_q, d_v)
    """
    B, h, seq_q, d_k = Q.shape
    d_v = V.size(-1)

    output = torch.zeros(B, h, seq_q, d_v, device=Q.device, dtype=Q.dtype)  # (B, h, seq_q, d_v)
    scale = math.sqrt(d_k)

    for start in range(0, seq_q, chunk_size):
        end = min(start + chunk_size, seq_q)
        Q_chunk = Q[:, :, start:end, :]  # (B, h, chunk, d_k)

        # Compute attention for this chunk of queries against ALL keys
        scores = Q_chunk @ K.transpose(-2, -1) / scale  # (B, h, chunk, seq_k)
        attn_weights = torch.softmax(scores, dim=-1)     # (B, h, chunk, seq_k)
        output[:, :, start:end, :] = attn_weights @ V    # (B, h, chunk, d_v)

    return output  # (B, h, seq_q, d_v)


# --- Verify correctness ---
torch.manual_seed(42)
B, h, seq_len, d_k = 2, 4, 512, 32
Q = torch.randn(B, h, seq_len, d_k)
K = torch.randn(B, h, seq_len, d_k)
V = torch.randn(B, h, seq_len, d_k)

out_chunked = chunked_attention(Q, K, V, chunk_size=64)
out_naive = naive_attention(Q, K, V)

print(f"Chunked output shape: {out_chunked.shape}")
print(f"Max diff vs naive: {(out_chunked - out_naive).abs().max():.2e}")
assert torch.allclose(out_chunked, out_naive, atol=1e-5)
print("Chunked attention matches naive attention.")

# Memory comparison
print(f"\nPeak attention matrix memory (B={B}, h={h}, seq={seq_len}):")
naive_mb = B * h * seq_len * seq_len * 4 / (1024 ** 2)
chunk_size = 64
chunked_mb = B * h * chunk_size * seq_len * 4 / (1024 ** 2)
print(f"  Naive:   {naive_mb:.1f} MB  (seq x seq)")
print(f"  Chunked: {chunked_mb:.1f} MB  (chunk_size x seq)")
print(f"  Reduction: {naive_mb / chunked_mb:.1f}x")

---
## 3.8 -- Full Transformer Block

The standard transformer block combines everything from this module:

```
         x
         |
    [LayerNorm]      <-- pre-norm
         |
    [Multi-Head Attention]
         |
       + x           <-- residual connection
         |
    [LayerNorm]
         |
    [Feed-Forward Network: Linear -> SiLU/GELU -> Linear]
         |
       + x           <-- residual connection
         |
       output
```

**Feed-Forward Network (FFN):** Two linear layers with a nonlinear activation between them. The hidden dimension is typically `4 * d_model`, creating an "expansion-contraction" bottleneck.

**Activation choices:** Modern transformers use **SiLU** (Swish) or **GELU** rather than ReLU. Both are smooth approximations that avoid the hard zero of ReLU.

### Worked Example: TransformerBlock with Pre-Norm + Gradient Flow Verification

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network.

    Args:
        d_model: Input/output dimension.
        d_ff: Hidden dimension (typically 4 * d_model).
    """

    def __init__(self, d_model: int, d_ff: Optional[int] = None) -> None:
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, seq, d_model) -> (B, seq, d_model)"""
        return self.net(x)


class TransformerBlock(nn.Module):
    """Standard pre-norm transformer block.

    Args:
        d_model: Model dimension.
        num_heads: Number of attention heads.
        d_ff: FFN hidden dimension (default: 4 * d_model).
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: Optional[int] = None) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply transformer block.

        Args:
            x: (B, seq, d_model)
        Returns:
            (B, seq, d_model)
        """
        # Pre-norm self-attention + residual
        x_norm = self.norm1(x)                                    # (B, seq, d_model)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)          # (B, seq, d_model)
        x = x + attn_out                                          # (B, seq, d_model)

        # Pre-norm FFN + residual
        x = x + self.ffn(self.norm2(x))                           # (B, seq, d_model)

        return x  # (B, seq, d_model)


# --- Test: stack 4 blocks, verify gradient flow ---
torch.manual_seed(42)
d_model, num_heads, n_blocks = 64, 8, 4

blocks = nn.Sequential(*[TransformerBlock(d_model, num_heads) for _ in range(n_blocks)])

x = torch.randn(2, 20, d_model, requires_grad=True)
output = blocks(x)

# Check output
print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")
print(f"Total parameters: {sum(p.numel() for p in blocks.parameters()):,}")

# Verify gradient flow through all 4 blocks
loss = output.sum()
loss.backward()

print(f"\nGradient flow through {n_blocks} blocks:")
print(f"  Input gradient norm: {x.grad.norm():.4f}")
print(f"  Input gradient is finite: {x.grad.isfinite().all()}")

# Check gradients at each block
for i, block in enumerate(blocks):
    grad_norm = block.attn.W_Q.weight.grad.norm()
    print(f"  Block {i} W_Q gradient norm: {grad_norm:.4f}")

### Exercise 3.8: Minimal Vision Transformer (ViT) on CIFAR-10

Build a minimal ViT from scratch:
1. **Patch embedding:** Split image into patches, project each patch to `d_model`
2. **Positional encoding:** Add learned or sinusoidal PE
3. **Transformer blocks:** Stack N blocks
4. **Classification head:** Global average pool (or CLS token) -> linear -> logits

Train on CIFAR-10 to verify everything works end-to-end.

In [ ]:
# Exercise 3.8 -- Your implementation here

class PatchEmbedding(nn.Module):
    """Split image into patches and project to d_model."""

    def __init__(self, img_size: int, patch_size: int, in_channels: int, d_model: int) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, num_patches, d_model)
        pass


class MiniViT(nn.Module):
    """Minimal Vision Transformer for classification."""

    def __init__(
        self,
        img_size: int = 32,
        patch_size: int = 4,
        in_channels: int = 3,
        d_model: int = 128,
        num_heads: int = 4,
        num_blocks: int = 4,
        num_classes: int = 10,
    ) -> None:
        super().__init__()
        # TODO: implement
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, num_classes)
        pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class PatchEmbedding(nn.Module):
    """Split image into non-overlapping patches and project to d_model.

    Uses a Conv2d with kernel_size=stride=patch_size for efficient implementation.

    Args:
        img_size: Input image size (assumes square).
        patch_size: Size of each patch.
        in_channels: Number of input channels.
        d_model: Output embedding dimension.
    """

    def __init__(self, img_size: int, patch_size: int, in_channels: int, d_model: int) -> None:
        super().__init__()
        assert img_size % patch_size == 0
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size

        # Conv2d with kernel_size=stride=patch_size extracts and projects patches in one op
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Extract and embed patches.

        Args:
            x: (B, C, H, W)
        Returns:
            (B, num_patches, d_model)
        """
        x = self.proj(x)              # (B, d_model, H/P, W/P)
        x = x.flatten(2)              # (B, d_model, num_patches)
        x = x.transpose(1, 2)         # (B, num_patches, d_model)
        return x


class MiniViT(nn.Module):
    """Minimal Vision Transformer for classification.

    Architecture: Patch Embedding -> Positional Encoding -> N x TransformerBlock
                  -> LayerNorm -> Global Average Pool -> Linear -> Logits

    Args:
        img_size: Input image size (square).
        patch_size: Patch size.
        in_channels: Number of input channels.
        d_model: Transformer model dimension.
        num_heads: Number of attention heads per block.
        num_blocks: Number of transformer blocks.
        num_classes: Number of output classes.
    """

    def __init__(
        self,
        img_size: int = 32,
        patch_size: int = 4,
        in_channels: int = 3,
        d_model: int = 128,
        num_heads: int = 4,
        num_blocks: int = 4,
        num_classes: int = 10,
    ) -> None:
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        num_patches = self.patch_embed.num_patches

        # Learned positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, d_model) * 0.02)

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_model, num_heads) for _ in range(num_blocks)]
        )

        # Classification head
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: (B, C, H, W)
        Returns:
            (B, num_classes)
        """
        x = self.patch_embed(x)           # (B, num_patches, d_model)
        x = x + self.pos_embed            # (B, num_patches, d_model)
        x = self.blocks(x)                # (B, num_patches, d_model)
        x = self.norm(x)                  # (B, num_patches, d_model)
        x = x.mean(dim=1)                 # (B, d_model) -- global average pool
        x = self.head(x)                  # (B, num_classes)
        return x


# --- Quick test ---
torch.manual_seed(42)
vit = MiniViT(img_size=32, patch_size=4, in_channels=3, d_model=128, num_heads=4, num_blocks=4, num_classes=10)

dummy = torch.randn(4, 3, 32, 32)
logits = vit(dummy)
print(f"Input:  {dummy.shape}")     # (4, 3, 32, 32)
print(f"Output: {logits.shape}")    # (4, 10)
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")
print(f"Num patches: {vit.patch_embed.num_patches} (={32//4}x{32//4})")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(42)

# --- Data ---
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

# --- Model ---
vit = MiniViT(
    img_size=32, patch_size=4, in_channels=3,
    d_model=128, num_heads=4, num_blocks=4, num_classes=10
).to(device)

optimizer = torch.optim.AdamW(vit.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

print(f"Training MiniViT on CIFAR-10 ({device})")
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")

# --- Train for a few epochs (enough to see it learns) ---
num_epochs = 5
for epoch in range(num_epochs):
    vit.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        logits = vit(images)               # (B, 10)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    scheduler.step()
    train_acc = correct / total * 100
    avg_loss = running_loss / total

    # Quick eval
    vit.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            logits = vit(images)
            test_correct += (logits.argmax(dim=1) == labels).sum().item()
            test_total += images.size(0)
    test_acc = test_correct / test_total * 100

    print(f"  Epoch {epoch+1}/{num_epochs} -- Loss: {avg_loss:.4f}, "
          f"Train Acc: {train_acc:.1f}%, Test Acc: {test_acc:.1f}%")

print(f"\nFinal test accuracy: {test_acc:.1f}%")
print("The model learns, confirming all components work correctly."
      " A larger model with more epochs would reach ~85%+.")

---
## Capstone Exercise: Multi-Head Self-Attention on Image Patches

This capstone ties together everything from the module. You will:

1. **Patch** a CIFAR-10 image into 4x4 patches
2. **Flatten** patches into a sequence
3. **Add 2D positional encoding**
4. **Apply multi-head self-attention** (from scratch)
5. **Visualize attention patterns** to verify they make spatial sense (nearby patches should attend to each other)

This is a direct warmup for how attention is used inside diffusion U-Nets, where spatial feature maps are reshaped into sequences, self-attention is applied, and the result is reshaped back.

In [ ]:
# Capstone Exercise -- Your implementation here

# 1. Load a CIFAR-10 image
# 2. Patch it into 4x4 patches (8x8 grid for 32x32 image)
# 3. Flatten each patch and project to d_model
# 4. Add 2D positional encoding
# 5. Apply multi-head self-attention
# 6. Visualize: for a chosen query patch, show which patches it attends to most

# TODO: implement all steps

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

torch.manual_seed(42)

# ============================================================
# Step 1: Load a CIFAR-10 image
# ============================================================
transform_raw = transforms.ToTensor()
cifar = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_raw)
img, label = cifar[0]  # (3, 32, 32)
class_names = cifar.classes
print(f"Image shape: {img.shape}, Label: {class_names[label]}")

# ============================================================
# Step 2: Patch into 4x4 patches -> 8x8 grid = 64 patches
# ============================================================
patch_size = 4
C, H, W = img.shape
grid_h, grid_w = H // patch_size, W // patch_size
num_patches = grid_h * grid_w

# Use unfold to extract patches
# (C, H, W) -> (C, grid_h, patch_size, grid_w, patch_size) -> rearrange
patches = img.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
# patches shape: (C, grid_h, grid_w, patch_size, patch_size)
patches = patches.contiguous().view(C, grid_h, grid_w, patch_size, patch_size)
patches = patches.permute(1, 2, 0, 3, 4)  # (grid_h, grid_w, C, patch_size, patch_size)
patches_flat = patches.reshape(num_patches, -1)  # (64, C*patch_size*patch_size) = (64, 48)

print(f"Grid: {grid_h}x{grid_w} = {num_patches} patches")
print(f"Each patch: {C}x{patch_size}x{patch_size} = {patches_flat.shape[1]} dims")

# Visualize the patches
fig, axes = plt.subplots(grid_h, grid_w, figsize=(8, 8))
for r in range(grid_h):
    for c in range(grid_w):
        patch_img = patches[r, c].permute(1, 2, 0).numpy()  # (patch_size, patch_size, C)
        axes[r, c].imshow(patch_img)
        axes[r, c].axis("off")
fig.suptitle(f"Image split into {grid_h}x{grid_w} patches (class: {class_names[label]})")
plt.tight_layout()
plt.show()

# ============================================================
# Step 3: Project patches to d_model
# ============================================================
d_model = 64
patch_dim = C * patch_size * patch_size  # 48

patch_proj = nn.Linear(patch_dim, d_model)
patch_embeddings = patch_proj(patches_flat.unsqueeze(0))  # (1, 64, d_model)
print(f"\nPatch embeddings: {patch_embeddings.shape}")  # (1, 64, 64)

# ============================================================
# Step 4: Add 2D positional encoding
# ============================================================
pe_2d = PositionalEncoding2D(grid_size=grid_h, d_model=d_model)
patch_embeddings = pe_2d(patch_embeddings)  # (1, 64, d_model)
print(f"After 2D PE: {patch_embeddings.shape}")

# ============================================================
# Step 5: Apply multi-head self-attention
# ============================================================
num_heads = 4
mha_capstone = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

attn_output, attn_weights = mha_capstone(
    patch_embeddings, patch_embeddings, patch_embeddings
)
# attn_output: (1, 64, d_model)
# attn_weights: (1, num_heads, 64, 64)
print(f"Attention output: {attn_output.shape}")
print(f"Attention weights: {attn_weights.shape}")

# ============================================================
# Step 6: Visualize attention patterns
# ============================================================
# For several query patches, show which patches they attend to
query_patches = [0, 27, 36, 63]  # corners and center-ish
query_labels = ["top-left (0,0)", "middle-left (3,3)", "middle-right (4,4)", "bottom-right (7,7)"]

fig, axes = plt.subplots(len(query_patches), num_heads + 1, figsize=(16, 12))

for row_idx, (q_idx, q_label) in enumerate(zip(query_patches, query_labels)):
    # Show which patch we are querying
    q_r, q_c = q_idx // grid_w, q_idx % grid_w
    query_img = patches[q_r, q_c].permute(1, 2, 0).numpy()
    axes[row_idx, 0].imshow(query_img)
    axes[row_idx, 0].set_title(f"Query: {q_label}", fontsize=9)
    axes[row_idx, 0].axis("off")

    # Show attention weights for each head, reshaped to spatial grid
    for head in range(num_heads):
        weights_for_query = attn_weights[0, head, q_idx].detach().numpy()  # (64,)
        weights_grid = weights_for_query.reshape(grid_h, grid_w)  # (8, 8)

        im = axes[row_idx, head + 1].imshow(weights_grid, cmap="hot", vmin=0)
        axes[row_idx, head + 1].set_title(f"Head {head}", fontsize=9)
        axes[row_idx, head + 1].axis("off")

        # Mark the query position
        axes[row_idx, head + 1].plot(q_c, q_r, "cs", markersize=6)

fig.suptitle("Attention weights by query patch and head\n(cyan square = query position)", fontsize=13)
plt.tight_layout()
plt.show()

print("\nCapstone complete.")
print("Key observations:")
print("  - Different heads learn different attention patterns")
print("  - At initialization (random weights), patterns are relatively uniform")
print("  - After training, nearby patches tend to attend to each other")
print("  - This is exactly how spatial self-attention works in diffusion U-Nets")

---

## Summary & Connection to Diffusion Models

In this module, we built up the **attention mechanism** from first principles:

1. **Dot-product similarity** — the foundation of how tokens "look at" each other, used in every transformer-based diffusion model (Stable Diffusion, DiT, etc.)
2. **Scaled dot-product attention** — prevents gradient issues in high dimensions; the exact operation inside `F.scaled_dot_product_attention`
3. **Multi-head attention** — lets the model attend to different relationships simultaneously (spatial structure, color patterns, semantic content)
4. **Causal masking & cross-attention** — causal masking powers autoregressive generation; cross-attention is how diffusion models condition on text prompts (connecting CLIP/T5 embeddings to the image denoiser)
5. **Positional encoding** — since attention is permutation-invariant, positional encodings give the model a sense of order; in diffusion models, sinusoidal encodings also represent the **noise timestep**

These components are the building blocks of the **U-Net attention layers** (Module 04) and the **Diffusion Transformer (DiT)** architecture. Every modern image generation model uses multi-head cross-attention to fuse text and image representations.

Next up: **Module 04 — U-Net Architecture**, where we assemble convolutions and attention into the denoising backbone.